# Response Analysis Template

This notebook provides a template for analyzing response fractions from the unified statistics pipeline.

**Usage:**
1. Select statistic and source redshift
2. Load data from `stats.h5` files
3. Compute response fractions
4. Generate figures

**Outputs:**
- Scale-dependent response F_S(k) or F_S(ℓ)
- Cumulative response summary
- Tile heatmaps
- Marginal distributions
- Redshift evolution
- Additivity analysis

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import h5py
from pathlib import Path

# Add scripts to path
sys.path.insert(0, '../scripts')

from constants import STATS_BASE, MASS_THRESHOLDS_FLOAT, RADII
from response_utils import compute_F_S, compute_Delta_F, compute_epsilon, bootstrap_F

plt.style.use('default')
%matplotlib inline

## 1. Configuration

Select which statistic and redshift to analyze.

In [ ]:
# Select statistic: 'Pk', 'Cl', 'peaks', 'minima', 'pdf'
STATISTIC = 'peaks'

# For convergence stats, select source redshift index (0-3)
# 0: z~0.5, 1: z~1.0, 2: z~2.0, 3: z~2.5
Z_IDX = 1  # z~1.0

# Output directory for figures
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

print(f"Analyzing: {STATISTIC}")
if STATISTIC != 'Pk':
    print(f"Source redshift index: {Z_IDX}")

## 2. Load Data

Load statistics from HDF5 files for DMO, Hydro, and Replace models.

In [ ]:
def load_statistic(model, stat_name, z_idx=None):
    """Load statistic from stats.h5 file."""
    path = Path(STATS_BASE) / model / 'stats.h5'
    
    if not path.exists():
        print(f"WARNING: File not found: {path}")
        return None, None
    
    with h5py.File(path, 'r') as f:
        data = f[stat_name][:]
        
        # Get x-axis (k or ell or S/N)
        if stat_name == 'Pk':
            x = f['k_bins'][:]
        elif stat_name == 'Cl':
            x = f['ell_bins'][:]
        else:  # peaks, minima, pdf
            x = f['sn_bin_edges'][:]
            x = 0.5 * (x[:-1] + x[1:])  # Bin centers
        
        # Extract redshift slice for convergence stats
        if stat_name != 'Pk' and z_idx is not None:
            data = data[:, z_idx, :]  # (N_real, N_bins)
    
    return data, x


# Load DMO and Hydro
print("Loading DMO...")
S_dmo, x_bins = load_statistic('dmo', STATISTIC, Z_IDX)

print("Loading Hydro...")
S_hydro, _ = load_statistic('hydro', STATISTIC, Z_IDX)

if S_dmo is None or S_hydro is None:
    raise ValueError("Could not load DMO or Hydro data!")

print(f"Data shape: {S_dmo.shape}")
print(f"X-axis shape: {x_bins.shape}")

## 3. Load Replace Models

Load cumulative and discrete tile models.

In [ ]:
from constants import build_model_name, DISCRETE_MASS_BINS, DISCRETE_RADIUS_BINS

# Load cumulative models (4 mass × 4 radii = 16)
cumulative_data = {}
for mass in MASS_THRESHOLDS_FLOAT:
    for radius in RADII:
        model_name = build_model_name(mass, 1.00e15, 0.0, radius)
        data, _ = load_statistic(model_name, STATISTIC, Z_IDX)
        if data is not None:
            cumulative_data[(mass, radius)] = data

print(f"Loaded {len(cumulative_data)} cumulative models")

# Load discrete tile models (4 mass × 4 radii = 16)
tile_data = {}
for Ml, Mu in DISCRETE_MASS_BINS:
    for Ri, Ro in DISCRETE_RADIUS_BINS:
        model_name = build_model_name(Ml, Mu, Ri, Ro)
        data, _ = load_statistic(model_name, STATISTIC, Z_IDX)
        if data is not None:
            tile_data[(Ml, Mu, Ri, Ro)] = data

print(f"Loaded {len(tile_data)} tile models")

## 4. Compute Response Fractions

Calculate F_S for all models.

In [ ]:
# For density stats: average over LPs and snapshots
# For convergence stats: sum over realizations for bootstrap

if STATISTIC == 'Pk':
    # Average over LPs and snapshots
    S_dmo_mean = np.nanmean(S_dmo, axis=(0, 1))
    S_hydro_mean = np.nanmean(S_hydro, axis=(0, 1))
    
    # Compute response for each model
    F_cumulative = {}
    for (mass, radius), data in cumulative_data.items():
        S_replace_mean = np.nanmean(data, axis=(0, 1))
        F = compute_F_S(S_replace_mean, S_dmo_mean, S_hydro_mean)
        F_cumulative[(mass, radius)] = F
    
    F_tiles = {}
    for key, data in tile_data.items():
        S_tile_mean = np.nanmean(data, axis=(0, 1))
        F = compute_Delta_F(S_tile_mean, S_dmo_mean, S_hydro_mean)
        F_tiles[key] = F

else:
    # Use bootstrap for convergence stats
    F_cumulative = {}
    F_cumulative_err = {}
    
    for (mass, radius), data in cumulative_data.items():
        # Reshape for bootstrap: (N_real, N_bins)
        F_mean, F_std = bootstrap_F(data, S_dmo, S_hydro, n_bootstrap=1000)
        F_cumulative[(mass, radius)] = F_mean
        F_cumulative_err[(mass, radius)] = F_std
    
    F_tiles = {}
    F_tiles_err = {}
    
    for key, data in tile_data.items():
        F_mean, F_std = bootstrap_F(data, S_dmo, S_hydro, n_bootstrap=1000)
        F_tiles[key] = F_mean
        F_tiles_err[key] = F_std

print("Response fractions computed.")

## 5. Figure 1: Scale-Dependent Response

Plot F_S(k) or F_S(ℓ) for selected cumulative models.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, mass in enumerate(MASS_THRESHOLDS_FLOAT):
    ax = axes[i]
    
    for radius in RADII:
        if (mass, radius) in F_cumulative:
            F = F_cumulative[(mass, radius)]
            
            if STATISTIC == 'Pk':
                ax.semilogx(x_bins, F, label=f'α={radius}', marker='o', markersize=3)
            else:
                # With error bars
                F_err = F_cumulative_err.get((mass, radius), None)
                if F_err is not None:
                    ax.errorbar(x_bins, F, yerr=F_err, label=f'α={radius}', 
                               marker='o', markersize=3, capsize=2)
                else:
                    ax.plot(x_bins, F, label=f'α={radius}', marker='o', markersize=3)
    
    ax.axhline(0, color='k', linestyle='--', alpha=0.3)
    ax.axhline(1, color='k', linestyle='--', alpha=0.3)
    ax.set_xlabel('k [h/Mpc]' if STATISTIC == 'Pk' else ('ℓ' if STATISTIC == 'Cl' else 'S/N'))
    ax.set_ylabel('F_S')
    ax.set_title(f'M > {mass:.2e} M☉/h')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / f'{STATISTIC}_scale_dependent_response.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Figure 2: Cumulative Response Heatmap

Show mean response as function of mass threshold and radius.

In [ ]:
from response_utils import compute_mean_F

# Compute mean F for each model
F_grid = np.zeros((len(MASS_THRESHOLDS_FLOAT), len(RADII)))

for i, mass in enumerate(MASS_THRESHOLDS_FLOAT):
    for j, radius in enumerate(RADII):
        if (mass, radius) in F_cumulative:
            F = F_cumulative[(mass, radius)]
            F_mean = compute_mean_F(F, x_bins)
            F_grid[i, j] = F_mean
        else:
            F_grid[i, j] = np.nan

# Plot heatmap
fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(F_grid, aspect='auto', cmap='RdBu_r', vmin=0, vmax=1, origin='lower')

# Labels
mass_labels = [f'{m:.1e}' for m in MASS_THRESHOLDS_FLOAT]
radius_labels = [f'{r}' for r in RADII]

ax.set_xticks(range(len(RADII)))
ax.set_yticks(range(len(MASS_THRESHOLDS_FLOAT)))
ax.set_xticklabels(radius_labels)
ax.set_yticklabels(mass_labels)

ax.set_xlabel('Radius factor α (R₂₀₀)')
ax.set_ylabel('Mass threshold (M☉/h)')
ax.set_title(f'Mean Response Fraction: {STATISTIC}')

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('F_S')

# Annotate values
for i in range(len(MASS_THRESHOLDS_FLOAT)):
    for j in range(len(RADII)):
        text = ax.text(j, i, f'{F_grid[i, j]:.2f}',
                      ha="center", va="center", color="black", fontsize=10)

plt.tight_layout()
plt.savefig(FIG_DIR / f'{STATISTIC}_cumulative_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Figure 3: Discrete Tile Contributions

Show ΔF_S for each discrete tile.

In [ ]:
# Compute mean ΔF for each tile
n_mass_bins = len(DISCRETE_MASS_BINS)
n_radius_bins = len(DISCRETE_RADIUS_BINS)
Delta_F_grid = np.zeros((n_mass_bins, n_radius_bins))

for i, (Ml, Mu) in enumerate(DISCRETE_MASS_BINS):
    for j, (Ri, Ro) in enumerate(DISCRETE_RADIUS_BINS):
        key = (Ml, Mu, Ri, Ro)
        if key in F_tiles:
            F = F_tiles[key]
            F_mean = compute_mean_F(F, x_bins)
            Delta_F_grid[i, j] = F_mean
        else:
            Delta_F_grid[i, j] = np.nan

# Plot heatmap
fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(Delta_F_grid, aspect='auto', cmap='viridis', origin='lower')

# Labels
mass_labels = [f'{Ml:.1e}-{Mu:.1e}' for Ml, Mu in DISCRETE_MASS_BINS]
radius_labels = [f'{Ri}-{Ro}' for Ri, Ro in DISCRETE_RADIUS_BINS]

ax.set_xticks(range(n_radius_bins))
ax.set_yticks(range(n_mass_bins))
ax.set_xticklabels(radius_labels)
ax.set_yticklabels(mass_labels, fontsize=8)

ax.set_xlabel('Radius shell (R₂₀₀)')
ax.set_ylabel('Mass bin (M☉/h)')
ax.set_title(f'Tile Contributions ΔF_S: {STATISTIC}')

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('ΔF_S')

# Annotate values
for i in range(n_mass_bins):
    for j in range(n_radius_bins):
        text = ax.text(j, i, f'{Delta_F_grid[i, j]:.2f}',
                      ha="center", va="center", color="white", fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / f'{STATISTIC}_tile_contributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Additional Analyses

Add cells here for:
- Marginal distributions (integrate over mass or radius)
- Redshift evolution (loop over Z_IDX)
- Additivity analysis (compute ε = F_cumulative - Σ ΔF_tiles)
- Summary statistics tables